# Tuning value CNN

In [ ]:
import sys
sys.path.append("..")

import optuna
import torch
from torch import nn
from torch.utils import data
from tqdm import tqdm
from matplotlib import pyplot as plt

from ai.dataset import EvalDataset
from ai.model import ValueCNN

## Hyperparameters

In [ ]:
batch_size = 256
num_trials = 30
tune_epochs = 2
tune_train_size = 300_000
tune_val_size = 50_000
max_cost = 2_500_000

## Data

In [ ]:
dataset = EvalDataset("../data/evals.csv", "../data/evals.pt")
train_dataset, val_dataset, test_dataset = data.random_split(dataset, [0.8, 0.1, 0.1], generator=torch.Generator().manual_seed(42))

tune_train = data.Subset(train_dataset, range(tune_train_size))
tune_val = data.Subset(val_dataset, range(tune_val_size))
tune_train_loader = data.DataLoader(tune_train, batch_size=batch_size, shuffle=True)
tune_val_loader = data.DataLoader(tune_val, batch_size=batch_size, shuffle=False)

print("Tune train:", len(tune_train), " tune val:", len(tune_val))

## Training functions

In [ ]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
loss_fn = nn.MSELoss()

In [ ]:
def train_one_epoch(model, optimizer, train_loader, loss_fn):
    model.train()
    total_loss, num_batches = 0., 0

    for data_inputs, data_labels in tqdm(train_loader, leave=False):
        data_inputs = data_inputs.to(device)
        data_labels = data_labels.to(device)

        preds = model(data_inputs)
        loss = loss_fn(preds, data_labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        num_batches += 1

    return total_loss / num_batches

In [ ]:
def eval_model(model, data_loader, loss_fn):
    model.eval()
    total_loss, num_batches = 0., 0
    total_abs_error, num_preds = 0., 0

    with torch.no_grad():
        for data_inputs, data_labels in tqdm(data_loader, leave=False):
            data_inputs, data_labels = data_inputs.to(device), data_labels.to(device)
            preds = model(data_inputs)

            total_loss += loss_fn(preds, data_labels).item()
            num_batches += 1

            total_abs_error += (preds - data_labels).abs().sum().item()
            num_preds += data_labels.shape[0]

    return total_loss / num_batches, total_abs_error / num_preds

## Objective

In [ ]:
def objective(trial):
    num_conv_layers = trial.suggest_int("num_conv_layers", 2, 5)
    num_filters = trial.suggest_categorical("num_filters", [16, 32, 64])
    num_layers = trial.suggest_int("num_layers", 1, 5)
    num_hidden = trial.suggest_int("num_hidden", 64, 512, log=True)
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True)

    cost = 7 * num_filters * 9 * 64 + (num_conv_layers - 1) * num_filters * num_filters * 9 * 64 + num_filters * 64 * num_hidden + (num_layers - 1) * num_hidden * num_hidden
    if cost > max_cost:
        raise optuna.TrialPruned()

    model = ValueCNN(num_channels=7, num_filters=num_filters, num_conv_layers=num_conv_layers, num_hidden=num_hidden, num_layers=num_layers)
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    for epoch in range(tune_epochs):
        train_one_epoch(model, optimizer, tune_train_loader, loss_fn)
        val_loss, val_mae = eval_model(model, tune_val_loader, loss_fn)
        trial.report(val_loss, epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()

    trial.set_user_attr("val_mae", val_mae)
    trial.set_user_attr("cost", cost)
    return val_loss

## Search

In [ ]:
study = optuna.create_study(direction="minimize", pruner=optuna.pruners.MedianPruner())
study.optimize(objective, n_trials=num_trials)

print("Best params:", study.best_params)
print("Best val loss:", study.best_value)
print("Best val MAE:", study.best_trial.user_attrs["val_mae"])

## Results

In [ ]:
optuna.visualization.matplotlib.plot_optimization_history(study)
plt.show()

optuna.visualization.matplotlib.plot_param_importances(study, evaluator=optuna.importance.PedAnovaImportanceEvaluator())
plt.show()